In [1]:
!pip install nltk

In [3]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg

corpus = gutenberg.raw('austen-emma.txt') + """
I love coding
AI is the future
machine learning is powerful
deep learning is amazing
"""

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


In [5]:
import nltk
from collections import defaultdict, Counter

nltk.download('punkt')
nltk.download('punkt_tab')

def preprocess(text):
    sentences = nltk.sent_tokenize(text.lower())
    tokenized = [nltk.word_tokenize(sent) for sent in sentences]
    return tokenized

tokens = preprocess(corpus)

print(tokens[:2])  # check sample

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


[['[', 'emma', 'by', 'jane', 'austen', '1816', ']', 'volume', 'i', 'chapter', 'i', 'emma', 'woodhouse', ',', 'handsome', ',', 'clever', ',', 'and', 'rich', ',', 'with', 'a', 'comfortable', 'home', 'and', 'happy', 'disposition', ',', 'seemed', 'to', 'unite', 'some', 'of', 'the', 'best', 'blessings', 'of', 'existence', ';', 'and', 'had', 'lived', 'nearly', 'twenty-one', 'years', 'in', 'the', 'world', 'with', 'very', 'little', 'to', 'distress', 'or', 'vex', 'her', '.'], ['she', 'was', 'the', 'youngest', 'of', 'the', 'two', 'daughters', 'of', 'a', 'most', 'affectionate', ',', 'indulgent', 'father', ';', 'and', 'had', ',', 'in', 'consequence', 'of', 'her', 'sister', "'s", 'marriage', ',', 'been', 'mistress', 'of', 'his', 'house', 'from', 'a', 'very', 'early', 'period', '.']]


In [6]:
def build_ngrams(tokens, n):
    ngrams = defaultdict(Counter)

    for sentence in tokens:
        sentence = ['<s>']*(n-1) + sentence + ['</s>']

        for i in range(len(sentence)-n+1):
            prefix = tuple(sentence[i:i+n-1])
            word = sentence[i+n-1]
            ngrams[prefix][word] += 1

    return ngrams

# Build models
unigram = build_ngrams(tokens, 1)
bigram = build_ngrams(tokens, 2)
trigram = build_ngrams(tokens, 3)

print("Models built successfully ✅")

Models built successfully ✅


In [7]:
def get_prob(counter, word, vocab_size):
    return (counter[word] + 1) / (sum(counter.values()) + vocab_size)

In [8]:
vocab = set([w for sent in tokens for w in sent])
vocab_size = len(vocab)

print("Vocab size:", vocab_size)

Vocab size: 7917


In [9]:
def autocomplete(text, top_k=5):
    words = nltk.word_tokenize(text.lower())

    # Try trigram
    if len(words) >= 2:
        prefix = tuple(words[-2:])
        if prefix in trigram:
            probs = {w: get_prob(trigram[prefix], w, vocab_size) for w in trigram[prefix]}
            return sorted(probs, key=probs.get, reverse=True)[:top_k]

    # Backoff to bigram
    if len(words) >= 1:
        prefix = tuple([words[-1]])
        if prefix in bigram:
            probs = {w: get_prob(bigram[prefix], w, vocab_size) for w in bigram[prefix]}
            return sorted(probs, key=probs.get, reverse=True)[:top_k]

    # Backoff to unigram
    probs = {w: get_prob(unigram[()], w, vocab_size) for w in unigram[()]}
    return sorted(probs, key=probs.get, reverse=True)[:top_k]

In [10]:
print("Input: i love")
print("Suggestions:", autocomplete("i love"))

print("\nInput: machine")
print("Suggestions:", autocomplete("machine"))

print("\nInput: deep learning")
print("Suggestions:", autocomplete("deep learning"))

Input: i love
Suggestions: ['to', 'so', 'an', 'every', 'coding']

Input: machine
Suggestions: ['learning']

Input: deep learning
Suggestions: ['is']


In [11]:
while True:
    text = input("\nEnter text (type 'exit' to stop): ")
    if text == "exit":
        break
    print("Suggestions:", autocomplete(text))


Enter text (type 'exit' to stop): ChatGPT helps
Suggestions: [',', '</s>', '.', 'the', 'to']

Enter text (type 'exit' to stop): autocomplete
Suggestions: [',', '</s>', '.', 'the', 'to']

Enter text (type 'exit' to stop): learning 
Suggestions: ['is']

Enter text (type 'exit' to stop): exit
